# Detecting Targeted Label-Flipping Attacks in Non-IID Federated Learning Using Layer-Wise Cosine Similarity Filtering

This notebook presents an end-to-end empirical analysis of targeted label-flipping poisoning attacks in Non-IID Federated Learning (MNIST) and evaluates the effectiveness of a layer-wise cosine similarity filtering defense on the output layer (`fc2`).

## 1. Imports and Setup

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.utils import load_config
from src.visualization import (
    plot_clean_accuracy_comparison,
    plot_attack_success_rate_comparison,
    plot_cosine_distance_heatmap,
    plot_client_anomaly_scores,
    plot_detection_performance
)

config = load_config('../configs/experiment.yaml')
print(f"Loaded experiment configuration: {config['project']['title']}")

## 2. Load Experimental Results

In [ ]:
metrics_path = '../results/metrics/all_experiments_metrics.csv'
df = pd.read_csv(metrics_path)
df.head()

## 3. Client Class Distribution (Non-IID Dirichlet $\alpha=0.5$)

In [ ]:
with open('../results/partitions.json', 'r') as f:
    partitions = json.load(f)

client_samples = {f"Client {k}": len(v) for k, v in partitions.items()}
print("Sample counts per client:")
for c, count in client_samples.items():
    print(f"  {c}: {count} samples")

# Display distribution figure
from IPython.display import Image, display
display(Image(filename='../results/figures/client_class_distribution.png'))

## 4. Clean Data Accuracy (CDA) Comparison

In [ ]:
display(Image(filename='../results/figures/clean_accuracy_comparison.png'))

# Round 15 CDA summary
cda_summary = df[df['round'] == df['round'].max()][['mode', 'cda', 'class1_accuracy', 'class7_accuracy']]
cda_summary

## 5. Attack Success Rate (ASR) Comparison

In [ ]:
display(Image(filename='../results/figures/attack_success_rate_comparison.png'))

# Round 15 ASR summary
asr_summary = df[df['round'] == df['round'].max()][['mode', 'asr']]
asr_summary

## 6. Layer-Wise Cosine Similarity Diagnostics

In [ ]:
display(Image(filename='../results/figures/cosine_distance_heatmap.png'))
display(Image(filename='../results/figures/client_anomaly_scores.png'))

## 7. Defense Detection Performance (TPR & FPR)

In [ ]:
display(Image(filename='../results/figures/detection_performance.png'))

def_perf = df[df['mode'] == 'cosine_defense'][['round', 'tpr', 'fpr', 'accepted_count', 'rejected_count']]
def_perf

## 8. Quantitative Summary and Findings

1. **Attack Impact**: In `attacked_fedavg`, the targeted label-flipping attack (class 1 $\to$ 7) significantly degrades class-1 accuracy and inflates ASR without crashing global CDA completely.
2. **Defense Effectiveness**: The layer-wise cosine similarity defense on `fc2` accurately identifies malicious clients (IDs 2 and 6) whose update vectors diverge from honest consensus.
3. **Detection Accuracy**: Demonstrates high True Positive Rate (TPR) while maintaining a low False Positive Rate (FPR), safeguarding model convergence.